In [ ]:
from gpaw import GPAW
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
import tbmodels 

# Only one field strength: 0.5
field_strength = [0.0] 
band_gaps_dft = []
band_gaps_tb = []

# Create a single figure and axis
fig, ax = plt.subplots(figsize=(8, 10))

for n in field_strength:
    # 1. Load DFT Calculation
    bs_filename = f'/home/jayf/gpaw_env/AAPRIME/BS_AAPRIME_RESULTS/AAPRIME_structure_GMKG_wfield_relaxation{n:.1f}.gpw'
    calc_bs = GPAW(bs_filename)
    
    # Load atoms for Fermi level logic
    atoms = read(bs_filename)
    calc_dft = atoms.calc
    fermi_level = calc_dft.get_fermi_level()
    
    # --- DFT Band Gap Calculation ---
    # We will find global VBM/CBM manually to be safe
    global_vbm_dft = -np.inf
    global_cbm_dft = np.inf
    
    num_kpts = len(calc_dft.get_ibz_k_points())
    num_spins = calc_dft.get_number_of_spins()

    for s in range(num_spins):
        for k in range(num_kpts):
            eig = calc_dft.get_eigenvalues(kpt=k, spin=s)
            
            occupied = eig[eig <= fermi_level]
            unoccupied = eig[eig > fermi_level]

            if len(occupied) > 0:
                current_vbm = np.max(occupied)
                if current_vbm > global_vbm_dft:
                    global_vbm_dft = current_vbm
            
            if len(unoccupied) > 0:
                current_cbm = np.min(unoccupied)
                if current_cbm < global_cbm_dft:
                    global_cbm_dft = current_cbm

    if global_vbm_dft != -np.inf and global_cbm_dft != np.inf:
        dft_gap = global_cbm_dft - global_vbm_dft
        if dft_gap < 0: dft_gap = 0.0
    else:
        dft_gap = np.nan
    
    band_gaps_dft.append(dft_gap)
    print(f"DFT Gap (E={n}): {dft_gap:.4f} eV")

    # --- Plot DFT Band Structure ---
    bs = calc_bs.band_structure()
    bs.plot(ax=ax, show=False, emax=10, label='DFT Bands' )

    # --- 2. TB Model Calculation & Overlay ---
    tb_gap = np.nan # Default if fail
    
    try:
        tb_filename = f'AAprime_hBN_bilayer_v3_{n:.1f}.hdf5'
        print(f"Loading TB model: {tb_filename}")
        model = tbmodels.Model.from_hdf5_file(tb_filename)

        # Get k-path
        path = bs.path
        k_points = path.kpts
        x_axis, _, _ = path.get_linear_kpoint_axis()

        # Calculate TB Eigenvalues
        tb_eigenvalues_list = model.eigenval(k_points) 
        tb_eigenvalues = np.array(tb_eigenvalues_list)

        # --- Calculate TB Band Gap ---
        # Assuming Band 7 is VBM and Band 8 is CBM (0-indexed)
        # We calculate the gap as the Global Min of Band 8 minus Global Max of Band 7
        vbm_idx = 7
        cbm_idx = 8
        
        tb_vbm_global = np.max(tb_eigenvalues[:, vbm_idx])
        tb_cbm_global = np.min(tb_eigenvalues[:, cbm_idx])
        
        tb_gap = tb_cbm_global - tb_vbm_global
        band_gaps_tb.append(tb_gap)
        print(f"TB Gap (E={n}):  {tb_gap:.4f} eV")

        # --- Align TB to DFT ---
        # Align using the VBMs calculated above
        align_shift = global_vbm_dft - tb_vbm_global
        tb_eigenvalues_shifted = tb_eigenvalues + align_shift
        print(align_shift)

        # --- Plot TB Bands ---
        ax.plot(x_axis, tb_eigenvalues_shifted[:, 0], color='red', linestyle='--', linewidth=1.5, label='TB Model') 
        ax.plot(x_axis, tb_eigenvalues_shifted[:, 1:], color='red', linestyle='--', linewidth=1.5)

    except Exception as e:
        print(f"TB Error: {e}")
        band_gaps_tb.append(np.nan)

    # --- Customize Plot with BOTH Gaps ---
    ax.set_ylim(-10, 10) # Zoomed in to see the gap better
    
    title_text = f'Band Structure at E = {n:.1f} V/Å\n'
    title_text += f'DFT Gap: {dft_gap:.4f} eV  |  TB Gap: {tb_gap:.4f} eV'
    
    ax.set_title(title_text, fontsize=20)
    ax.set_ylabel('Energy (eV)', fontsize=18)
    ax.set_xlabel('k-path', fontsize=18)
    ax.tick_params(labelsize=18)
    ax.legend(fontsize=22)


save_name = f'TB_AB_E{field_strength}.png'
plt.savefig(save_name, dpi=300)
plt.tight_layout()
plt.show()